# DORA Metrics for Deployment Frequency Only

## Objective

### What do we want to be able to show for deployment frequency?

- Total deployment frequency over time.
  - Regardless of appID, Env, or Status
- Then split by separately:
  - AppID
  - Env
  - Status
- Then maybe a matrix of these options
 - AppID / Env / Status

## Environment

In [1]:
print("Kernel is working.")

Kernel is working.


In [2]:
from dash import dcc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

## Data Ingestion

**Output:** pandas dataframe *df_deployments*.

In [3]:
# Useful reference: <https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html>
raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module2_project/doraview/data/json/deployment_frequency.json'

# Try reading with explicit encoding and error handling
try:
    df_deployments = pd.read_json(raw_file_path, encoding='utf-8', convert_dates=["deployed_at"])
    print("\nDataframe info:")
    print(df_deployments.info())
    print("\nFirst 5 rows:")
    print(df_deployments.head())
except Exception as e:
    print(f"Error: {str(e)}")


Dataframe info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   deployment_id     44 non-null     object             
 1   application_id    44 non-null     object             
 2   application_name  44 non-null     object             
 3   environment       44 non-null     object             
 4   deployed_at       44 non-null     datetime64[ns, UTC]
 5   status            44 non-null     object             
 6   version           44 non-null     object             
dtypes: datetime64[ns, UTC](1), object(6)
memory usage: 2.5+ KB
None

First 5 rows:
  deployment_id application_id application_name environment  \
0   df_55eeff30         app001     Auth Service  production   
1   df_952faf50         app001     Auth Service     staging   
2   df_45056a55         app001     Auth Service  production   
3   df_4a4be47

## Data Manipulation

Manipulate *df_deployments* into format needed to produce figures.

- New columns to use for grouping:
  - row quater
  - row month

In [4]:
# Add new row and quater columns
df_deployments["month"] = df_deployments["deployed_at"].dt.month
df_deployments["qtr"] = df_deployments["deployed_at"].dt.quarter

df_deployments.head(100)

,deployment_id,application_id,application_name,environment,deployed_at,status,version,month,qtr
0,df_55eeff30,app001,Auth Service,production,2025-08-31 02:05:00+00:00,success,v3.3.4,8,3
1,df_952faf50,app001,Auth Service,staging,2025-08-07 11:06:00+00:00,success,v3.7.0,8,3
2,df_45056a55,app001,Auth Service,production,2025-07-27 12:08:00+00:00,success,v2.0.3,7,3
3,df_4a4be475,app001,Auth Service,staging,2025-08-16 06:24:00+00:00,success,v3.8.4,8,3
4,df_9ddc9087,app001,Auth Service,staging,2025-07-29 03:29:00+00:00,success,v1.7.6,7,3
5,df_09f6a4a8,app001,Auth Service,production,2025-09-30 17:01:00+00:00,success,v1.3.6,9,3
6,df_0721d117,app001,Auth Service,production,2025-08-11 14:09:00+00:00,success,v2.1.9,8,3
7,df_3ad908f6,app001,Auth Service,staging,2025-08-07 03:03:00+00:00,success,v3.9.3,8,3
8,df_295ed3f1,app001,Auth Service,production,2025-07-07 16:35:00+00:00,success,v2.4.3,7,3
9,df_1df285b7,app001,Auth Service,staging,2025-08-09 09:38:00+00:00,success,v2.9.6,8,3


In [5]:
# Reduce number of columns and check data types carry over.
df_deploy_reduced = df_deployments[["application_id","application_name","environment","status","deployed_at","month","qtr"]]
df_deploy_reduced.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   application_id    44 non-null     object             
 1   application_name  44 non-null     object             
 2   environment       44 non-null     object             
 3   status            44 non-null     object             
 4   deployed_at       44 non-null     datetime64[ns, UTC]
 5   month             44 non-null     int32              
 6   qtr               44 non-null     int32              
dtypes: datetime64[ns, UTC](1), int32(2), object(4)
memory usage: 2.2+ KB


## Figures

### ChatGPT - Deployment Frequency (DF)

**Best Graph Type:** Bar Chart (vertical) or Line Chart — plotted over time

**Why:** Deployment Frequency is a count of deployments per day/week/month.

**You want to show:**

- How deployment cadence changes over time
- Peaks, slow periods, or stability
- Differences between environments (if applicable)

Bar charts make discrete counts very clear.
Line charts work well if you want to show smoothed trends across longer periods.

**Optional secondary views:**

- Heatmap Calendar (GitHub-style commit calendar)
- Grouped bars (different microservices or applications per time window)

### Styling

['#636EFA', # the plotly blue you can see above
 '#EF553B',
 '#00CC96',
 '#AB63FA',
 '#FFA15A',
 '#19D3F3',
 '#FF6692',
 '#B6E880',
 '#FF97FF',
 '#FECB52']

### Multi-Group Display

In [6]:
df_deploy_multi_group = df_deployments.groupby([
	"application_id",
	"application_name",
	"month",
	"environment",
	"status"
	])["month"].count().reset_index(name="count")

df_deploy_multi_group.head(100)

,application_id,application_name,month,environment,status,count
0,app001,Auth Service,7,production,success,3
1,app001,Auth Service,7,staging,success,3
2,app001,Auth Service,8,production,success,2
3,app001,Auth Service,8,staging,success,4
4,app001,Auth Service,9,production,success,2
5,app002,Inventory API,7,production,failed,1
6,app002,Inventory API,7,production,success,1
7,app002,Inventory API,8,staging,failed,1
8,app002,Inventory API,8,staging,success,4
9,app002,Inventory API,9,staging,failed,1


In [7]:
# display dataframe as figure
fig_month_multi_bar = px.bar(
	data_frame=df_deploy_multi_group,
	title="Total Monthly Deployments",	# Label for the figure.
	x="month",							# Column for use on x-axis
	y="count",							# Column for use on y-axis
	subtitle="All Applications"
)

fig_month_multi_bar.update_yaxes(
	title_text="Deployment Count"		# Update title for y-axis
)

fig_month_multi_bar.update_xaxes(
	title_text="Deployment Month",		# Update title for x-axis
	tickvals=list(range(1,13)),			# Specify values of x-axis
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']	# Allocate text to tick values.
)

fig_month_multi_bar.show()

### Monthly Deployments

In [8]:
# create grouped dataframe
df_deploy_group_month = df_deployments.groupby(["month"])["month"].count().reset_index(name="count")

df_deploy_group_month.head(100)

,month,count
0,7,12
1,8,22
2,9,10


In [9]:
# display dataframe as figure
fig_month_all_bar = px.bar(
	data_frame=df_deploy_group_month,
	title="Total Monthly Deployments",	# Label for the figure.
	x="month",							# Column for use on x-axis
	y="count",							# Column for use on y-axis
	subtitle="All Applications",
)

fig_month_all_bar.update_yaxes(
	title_text="Deployment Count"		# Update title for y-axis
)

fig_month_all_bar.update_xaxes(
	title_text="Deployment Month",		# Update title for x-axis
	tickvals=list(range(1,13)),			# Specify values of x-axis
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']	# Allocate text to tick values.
)

fig_month_all_bar.show()

In [10]:
# display dataframe as figure
fig_month_all_line = px.line(
	data_frame=df_deploy_group_month,
	title="Total Monthly Deployments",	# Label for the figure.
	x="month",							# Column for use on x-axis
	y="count",							# Column for use on y-axis
	subtitle="All Applications"
)

fig_month_all_line.update_yaxes(
	title_text="Deployment Count"		# Update title for y-axis
)

fig_month_all_line.update_xaxes(
	title_text="Deployment Month",		# Update title for x-axis
	tickvals=list(range(1,13)),			# Specify values of x-axis
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']	# Allocate text to tick values.
)

fig_month_all_line.show()

### Monthly Deployments by Application

In [11]:
# create grouped dataframe
df_deploy_group_month_app = df_deployments.groupby(["application_name","month"])["month"].count().reset_index(name="count")

df_deploy_group_month_app.head(100)

,application_name,month,count
0,Auth Service,7,6
1,Auth Service,8,6
2,Auth Service,9,2
3,Customer Portal,7,4
4,Customer Portal,8,11
5,Customer Portal,9,4
6,Inventory API,7,2
7,Inventory API,8,5
8,Inventory API,9,4


In [12]:
# display dataframe as figure
fig_month_all_bar = px.bar(
	data_frame=df_deploy_group_month_app,
	title="Total Monthly Deployments by Application",
	x="month",
	y="count",
	color="application_name",	# Changing this determines the divisions
)

fig_month_all_bar.update_yaxes(
	title_text="Deployment Count"
)

fig_month_all_bar.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_all_bar.show()

In [13]:
# display dataframe as figure
fig_month_all_line = px.line(
	data_frame=df_deploy_group_month_app,
	title="Total Monthly Deployments by Application",
	x="month",
	y="count",
	color="application_name"
)

fig_month_all_line.update_yaxes(
	title_text="Deployment Count"
)

fig_month_all_line.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_all_line.show()

### Monthly Deployments by Environment

In [14]:
# create grouped dataframe
df_deploy_group_month_env = df_deployments.groupby(["environment","month"])["month"].count().reset_index(name="count")

df_deploy_group_month_env.head(100)

,environment,month,count
0,production,7,7
1,production,8,9
2,production,9,4
3,staging,7,5
4,staging,8,13
5,staging,9,6


In [15]:
# display dataframe as figure
fig_month_env_bar = px.bar(
	data_frame=df_deploy_group_month_env,
	title="Total Deployments by Environment",
	x="month",
	y="count",
	color="environment",
	color_discrete_map={"staging":"#636EFA", "production":"#EF553B"},
)

fig_month_env_bar.update_yaxes(
	title_text="Deployment Count"
)

fig_month_env_bar.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_env_bar.show()

In [16]:
# display dataframe as figure
fig_month_env_line = px.line(
	data_frame= df_deploy_group_month_env,
	title="Total Deployments by Environment",
	x="month",
	y="count",
	color="environment",
	color_discrete_map={
		"staging":"#636EFA",
		"production":"#EF553B"
		},
)

fig_month_env_line.update_yaxes(
	title_text="Deployment Count"
)

fig_month_env_line.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_env_line.show()

### Monthly Deployments by Status

In [17]:
# create grouped dataframe
df_deploy_group_month_status = df_deployments.groupby(["status","month"])["month"].count().reset_index(name="count")

df_deploy_group_month_status.head(100)

,status,month,count
0,failed,7,2
1,failed,8,3
2,failed,9,2
3,success,7,10
4,success,8,19
5,success,9,8


In [18]:
# display dataframe as figure
fig_month_stat_bar = px.bar(
	df_deploy_group_month_status,
	title="Total Deployments by Status",
	x="month",
	y="count",
	color="status",
	color_discrete_map={
		"success":"#636EFA",
		"failed":"#EF553B"
		},
)

fig_month_stat_bar.update_yaxes(
	title_text="Deployment Count"
)

fig_month_stat_bar.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_stat_bar.show()

In [19]:
# display dataframe as figure
fig_month_stat_line = px.line(
	df_deploy_group_month_status,
	title="Total Deployments by Status",
	x="month",
	y="count",
	color="status",
	color_discrete_map={
		"success":"#636EFA",
		"failed":"#EF553B"
		},
)

fig_month_stat_line.update_yaxes(
	title_text="Deployment Count"
)

fig_month_stat_line.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_stat_line.show()